## Evaluate poloidal current sensitivities wrt Fx

In [ ]:
from utils import *
NET_DIR = 'data/3479268/'
CURR_MASK = np.arange(76, 95)

In [ ]:
ds = LiuqeDataset(EVAL_DS_PATH)
fg = ds.fg # Full grid
phys = ds.data[PHYS]
p = torch.stack([to_tensor(RRD).view(-1), to_tensor(ZZD).view(-1)], dim=1) 
xμσ = to_tensor(np.load(NET_DIR + 'x_mean_std.npz')['x_mean_std'], device=CPU)
μ, σ = xμσ[0,:].numpy(), xμσ[1,:].numpy()
net = liuqeML(xμσ)
net.load_state_dict(torch.load(NET_DIR + 'best_Fx.pth', map_location=CPU));

In [ ]:
#inference
dsi = np.random.randint(0, len(ds))

ph = phys[dsi] 
fx, iy, br, bz = fg[dsi]

r = p.clone()[:,0]; p.requires_grad = True
ph.requires_grad = True

fxp, _, iyp = net(ph.view(1,-1), p.view(1,-1,2)) 

# derivative wrt r and z
dfx_drz = torch.autograd.grad(outputs=fxp.sum(), inputs=p, create_graph=True)[0]
brp = -dfx_drz[:,1] / (2*π*r) # Br from derivative
bzp = dfx_drz[:,0] / (2*π*r)  # Bz from derivative

# derivative wrt physics inputs
dfx_dph = torch.autograd.grad(outputs=fxp.sum(), inputs=ph, create_graph=True)[0]

fxp, brp, bzp, iyp = fxp.view(-1), brp.view(-1), bzp.view(-1), iyp.view(-1)


plt.figure(figsize=(15,10))
fxp, brp, bzp, iyp, ph, dfx_dph = map(lambda x: x.view(-1).detach().numpy(), (fxp, brp, bzp, iyp, ph, dfx_dph))
plt.subplot(2,4,1), plot_vessel(), plt.axis('equal'), plt.title('Fx p')
plt.scatter(RRD, ZZD, c=fxp), plt.colorbar()
plt.subplot(2,4,2), plot_vessel(), plt.axis('equal'), plt.title('Br p')
plt.scatter(RRD, ZZD, c=brp), plt.colorbar()
plt.subplot(2,4,3), plot_vessel(), plt.axis('equal'), plt.title('Bz p')
plt.scatter(RRD, ZZD, c=bzp), plt.colorbar()
plt.subplot(2,4,4), plot_vessel(), plt.axis('equal'), plt.title('Iy p')
plt.scatter(RRD, ZZD, c=iyp), plt.colorbar()

# plt.subplot(2,4,5), plot_vessel(), plt.axis('equal'), plt.title('Fx')
# plt.scatter(RRD, ZZD, c=fx), plt.colorbar()
# plt.subplot(2,4,6), plot_vessel(), plt.axis('equal'), plt.title('Br')
# plt.scatter(RRD, ZZD, c=br), plt.colorbar()
# plt.subplot(2,4,7), plot_vessel(), plt.axis('equal'), plt.title('Bz')
# plt.scatter(RRD, ZZD, c=bz), plt.colorbar()
# plt.subplot(2,4,8), plot_vessel(), plt.axis('equal'), plt.title('Iy')
# plt.scatter(RRD, ZZD, c=iy), plt.colorbar()

plt.figure(figsize=(10,7))
plt.subplot(2,1,1), plt.title('Currents (A)'),
# plt.bar(CURR_MASK, ph[CURR_MASK])
plt.bar(range(len(ph)), (ph-μ)/σ)

plt.subplot(2,1,2), plt.title('Sensitivity (A)'),
plt.bar(range(len(dfx_dph)), (dfx_dph)*σ) # scale back to original units

plt.tight_layout(), plt.show()
